# Advanced RAG

A production-focused RAG implementation.

| Date | User | Change Type | Remarks |  
| ---- | ---- | ----------- | ------- |
| 11/03/2026   | Martin | Create  | Notebook created to explore a production designed RAG | 

# Content

* [Introduction](#introduction)

# Introduction

Contains these different techniques:

- Hybrid Search (BM25 + Dense Retrieval)
- MMR (Maximal Marginal Relevance) for diversity
  - A document has high marginal relevance if it is both relevant to the query and contains minimal similarity to previously selected documents. MMR is defined as
- Contextual Compression — strip irrelevant content from chunks
- Multi-Query Retrieval — generate query variants to beat phrasing sensitivity  
- Self-Querying Retrieval — LLM parses structured filters from natural language
- Reranking — cross-encoder reranks retrieved chunks by relevance
  - _Cross-encoder:_ Compares the similarity of chunks by combining two sentences where each token attends to each other token
- RAG Evaluation with faithfulness + answer relevance metrics

In [1]:
import os
from typing import List
from langchain_core.documents import Document

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [2]:
# ==============================
# Document + Embedding Loading
# ==============================
def load_and_chunk(data_dir: str, chunk_size=500, chunk_overlap=50):
  loader = DirectoryLoader(
    data_dir,
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'}
  )
  docs = loader.load()

  splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    add_start_index=True
  )
  return splitter.split_documents(docs)

def get_embeddings():
  return HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={"normalize_embeddings": True}
  )

def _format_docs(docs: List[Document]) -> str:
  """Collapse a list of Documents into a single context string for the prompt."""
  return "\n\n".join(doc.page_content for doc in docs)

RAG_PROMPT = ChatPromptTemplate.from_messages([
  ("system",
    "You are a helpful assistant. Answer using ONLY the context below. "
    "If the answer is not in the context, say you don't know.\n\nContext:\n{context}"),
  ("human", "{question}"),
])

## Hybrid Search

Run both methods and merge results. Elasticsearch does this internally

- _BM25:_ Okapi Best Matching 25. Ranking function used in search engines to estimate document relevance
- _Dense Retrieval:_ Uses embeddings to convert both query and documents to common space then perform a similarity search

PRODUCTION USE: Most enterprise RAG systems use hybrid search as the default.

In [3]:
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import Chroma
from langchain_core.runnables import RunnableLambda

In [4]:
def build_hybrid_retriever(chunks: List[Document], embeddings, k=3):
  print("\n[HYBRID] Building BM25 + Dense hybrid retriever...")

  # Sparse retriever: BM25
  bm25_retriever = BM25Retriever.from_documents(chunks)
  bm25_retriever.k = k

  # Dense retriever: Semantic search + cosine similairty
  vectorstore = Chroma.from_documents(chunks, embeddings)
  dense_retriever = vectorstore.as_retriever(search_kwargs={"k": k})

  def _merge(query: str) -> List[Document]:
    bm25_docs = bm25_retriever.invoke(query)
    dense_docs = dense_retriever.invoke(query)

    # Deduplication
    seen, merged = set(), []
    for doc in bm25_docs + dense_docs:
      if doc.page_content not in seen:
        seen.add(doc.page_content)
        merged.append(doc)
    return merged

  print("[HYBRID] Hybrid retriever ready (BM25 + Dense)")
  return RunnableLambda(_merge)

## Maximal Marginal Relevance (MMR)

Top-k similarity tends to return very similar documents. MMR aims to balance relevance and diversity. Each chunk is chosen based on its similarity to the query and dissimilarity to existing selected documents.

PRODUCTION USE: Always use MMR when k > 3 or your docs have high repetition.

In [5]:
def build_mmr_retriever(chunks: List[Document], embeddings, k=4, lambda_mult=0.5):
  print("\n[MMR] Building MMR retriever for diverse results...")

  vectorstore = Chroma.from_documents(chunks, embeddings)
  retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
      "k": k,
      "fetch_k": k * 4,            # Fetch more candidates, then diversify
      "lambda_mult": lambda_mult,  # 0.5 = balanced relevance/diversity
    }
  )
  print(f"[MMR] MMR retriever ready (k={k}, λ={lambda_mult})")
  return retriever

## Contextual Compression

__🥊 PROBLEM:__ Chunks can contain irrelevant information that waste context window

__💡 SOLUTION:__ Use an LLM to extract only the relevant portion. This reduces the noise sent to the final LLM

__PRODUCTION USE:__ Expensive (extra LLM call per chunk). Use when precision matters more than latency. Can be replaced with a smaller extractive model.

In [6]:
from langchain_core.runnables import RunnableLambda

In [7]:
def build_compression_retriever(base_retriever, llm):
  """
  Returns a Runnable that retrieves docs then compresses each one with the LLM.
  Output: List[Document] with page_content replaced by the extracted excerpt.
  """
  print("\n[COMPRESS] Building contextual compression retriever...")
  
  # Prompt to extract main context
  extract_prompt = ChatPromptTemplate.from_messages([
    ("system",
      "Extract only the sentences from the passage that are directly relevant "
      "to answering the question. Return ONLY the extracted text, nothing else. "
      "If nothing is relevant, return exactly: NO_RELEVANT_CONTENT",
    ),
    ("human", "Question: {question}\n\nPassage:\n{passage}")
  ])
  extractor_chain = extract_prompt | llm | StrOutputParser()

  def _compress(inputs: dict) -> List[Document]:
    query = inputs['question']
    docs = base_retriever.invoke(query)
    compressed = []
    for doc in docs:
      extracted = extractor_chain.invoke({
        "question": query,
        "passage": doc.page_content
      }).strip()
      if extracted and extracted != "NO_RELEVANT_CONTENT":
        compressed.append(Document(
          page_content=extracted,
          metadata=doc.metadata
        ))
    return compressed

  print("[COMPRESS] Compression retriever ready")
  return RunnableLambda(_compress)


## Multi-query Retrieval

__🥊 PROBLEM:__ RAG is sensitive to query phrasing. Completely different chunks can mean the same thing

__💡 SOLUTION:__ Use LLM to generate N paraphrases, retrieved for each, union results

__PRODUCTION USE:__ Adds latency (N LLM + retrieval calls) but improves recall. Good for chatbots

In [8]:
def build_multiquery_retriever(base_retriever, llm, n_queries: int = 3):
  """
  Returns a Runnable[str, List[Document]] that:
    1. Generates n_queries paraphrases of the input query
    2. Retrieves docs for each paraphrase
    3. Returns deduplicated union of all results
  """
  print("\n[MULTI-QUERY] Building multi-query retriever...")

  # Prompt to create N different phrasings
  query_gen_prompt = ChatPromptTemplate.from_messages([
    ("system",
      f"Generate exactly {n_queries} different phrasings of the user's question. "
      "Output one question per line, no numbering, no extra text."),
    ("human", "{question}"),
  ])
  query_gen_chain = query_gen_prompt | llm | StrOutputParser()

  def _multi_retrieve(query: str) -> List[Document]:
    raw = query_gen_chain.invoke(query)
    phrasings = [q.strip() for q in raw.strip().splitlines() if q.strip()]
    all_queries = [query] + phrasings # Include original query
    seen, results = set(), []
    for q in all_queries:
      for doc in base_retriever.invoke(q):
        if doc.page_content not in seen:
          seen.add(doc.page_content)
          results.append(doc)
    return results

  print(f"[MULTI-QUERY] Multi-query retriever ready ({n_queries} variants + original)")
  return RunnableLambda(_multi_retrieve)



## Metadata Filtering

__🥊 PROBLEM:__ All documents indexed together - domain bleed across queries

__💡 SOLUTION:__ Filter by metadata retrieval time

__PRODUCTION USE:__ For multi-tenant systmes and domain partitioning

In [9]:
def build_filtered_retriever(vectorstore, source_filter: str, k: int = 3):
  """
  Returns a retriever scoped to documents whose 'source' path contains source_filter.
  Swap the filter dict for any ChromaDB-supported $eq / $contains / $gte expression.
  """
  return vectorstore.as_retriever(
    search_kwargs={
      "k": k,
      "filter": {"source": {"$contains": source_filter}},
    }
  )


## Parent Document Retrieval

(Conceptual Implementation)

__🥊 PROBLEM:__ Small chunks are more precise but lose context. Big chunks provide context by are noisy

__💡 SOLUTION:__ Index small chunks for retrieval, return the larger chunks to the LLM. Best quality-per-token improvement

__PRODUCTION USE:__ Use langchain_core's InMemoryStore or Redis

## Query Routing

__🥊 PROBLEM:__ Vary the retrieval algorithm based on the type of query encountered (e.g factual lookups, code questions, etc.)

__💡 SOLUTION:__ A router (rule-based or LLM-based) selects the retrieval path

__PRODUCTION USE:__ Large-scale systems with multiple data sources


In [10]:
ROUTE_KEYWORDS = {
  "python":         ["python", "pandas", "numpy", "sklearn", "jupyter", "pip"],
  "ml_fundamentals":["supervised", "unsupervised", "overfitting", "regularization", "accuracy"],
  "llm_rag":        ["transformer", "embedding", "rag", "retrieval", "llm", "attention"],
}

def _keyword_router(query: str) -> str:
  q = query.lower()
  scores = {route: sum([1 for kw in kws if kw in q])
            for route, kws in ROUTE_KEYWORDS.items()}
  best = max(scores, key=scores.get)          

  print(f"  [ROUTER] '{query[:50]}' → {best} (scores: {scores})")
  return best

query_router = RunnableLambda(_keyword_router)

## RAG Evaluation

__🥊 PROBLEM:__ No way to monitor the performance of the RAG when pipeline changes

__💡 SOLUTION:__ Automated metrics covering retrieval quality and answer faithfulness

__PRODUCTION USE:__ Run on a golden eval dataset for every piepline change. 

In [11]:
def evaluate_retrieval(
  query: str,
  retrieved_chunks: List[Document],
  ground_truth: str) -> dict:
  """
  Lightweight keyword-overlap metrics. No LLM required.
  For production, replace with:

      from ragas import evaluate
      from ragas.metrics import faithfulness, answer_relevancy, context_precision
      result = evaluate(dataset, metrics=[faithfulness, answer_relevancy])
  """
  gt_keywords   = set(ground_truth.lower().split())
  context_words = set(" ".join(c.page_content.lower() for c in retrieved_chunks).split())
  coverage      = len(gt_keywords & context_words) / max(len(gt_keywords), 1)

  return {
    "keyword_coverage":    round(coverage, 3),
    "chunks_retrieved":    len(retrieved_chunks),
    "unique_sources":      len({c.metadata.get("source", "") for c in retrieved_chunks}),
    "avg_chunk_chars":     round(
      sum(len(c.page_content) for c in retrieved_chunks) / max(len(retrieved_chunks), 1)
    ),
  }

## Demo Runner

In [12]:
def run_advanced_rag_demo():
  print("\n" + "=" * 60)
  print("  ADVANCED RAG TECHNIQUES DEMO")
  print("=" * 60)

  data_dir = "./data"

  print("\n[SETUP] Loading and chunking documents...")
  chunks    = load_and_chunk(data_dir)
  embeddings = get_embeddings()
  vectorstore = Chroma.from_documents(chunks, embeddings)

  # LLM: Qwen3 0.6B via Ollama
  print("\n[LLM] Loading Qwen3 0.6B Model")
  from langchain_ollama import OllamaLLM
  llm = OllamaLLM(model="qwen3:0.6b", temperature=0)

  # ---- 1. Hybrid Search --------------------------------
  print("\n" + "-" * 50)
  print("TECHNIQUE 1: Hybrid Search (BM25 + Dense)")
  print("-" * 50)
  topic = "RMSE evaluation metrics"
  hybrid = build_hybrid_retriever(chunks, embeddings)
  results = hybrid.invoke(topic)
  print(f"  Retrieved {len(results)} docs (BM25 catches exact 'RMSE' match)")
  for r in results[:2]:
    print(f"  - [{os.path.basename(r.metadata.get('source', '?'))}] "
      f"{r.page_content[:80]}...")

  # ---- 2. MMR --------------------------------
  print("\n" + "-" * 50)
  print("TECHNIQUE 2: MMR — Diverse Retrieval")
  print("-" * 50)
  mmr = build_mmr_retriever(chunks, embeddings, k=4, lambda_mult=0.5)
  results = mmr.invoke("machine learning algorithms")
  print(f"  Retrieved {len(results)} diverse chunks:")
  for r in results:
    print(f"  - [{os.path.basename(r.metadata.get('source', '?'))}] "
      f"{r.page_content[:70]}...")

  # ---- 3. Contextual Compression --------------------------------
  # print("\n" + "-" * 50)
  # print("TECHNIQUE 3: Contextual Compression")
  # print("-" * 50)
  # base_retriever   = vectorstore.as_retriever(search_kwargs={"k": 3})
  # compress_retriever = build_compression_retriever(base_retriever, llm)
  # results = compress_retriever.invoke({"question": "What prevents overfitting?"})
  # print(f"  Compressed to {len(results)} relevant excerpts")
  # for r in results[:2]:
  #   print(f"  - {r.page_content[:100]}...")
  
  # ---- 4. Multi-Query --------------------------------
  print("\n" + "-" * 50)
  print("TECHNIQUE 4: Multi-Query Retrieval")
  print("-" * 50)
  base_retriever   = vectorstore.as_retriever(search_kwargs={"k": 3})
  mq_retriever = build_multiquery_retriever(base_retriever, llm, n_queries=3)
  results = mq_retriever.invoke("How do I stop my model from memorising training data?")
  print(f"  Retrieved {len(results)} unique docs across all query variants")

  # ---- 5. Metadata Filtering --------------------------------
  print("\n" + "-" * 50)
  print("TECHNIQUE 5: Metadata Filtering")
  print("-" * 50)
  try:
    filtered = build_filtered_retriever(vectorstore, "python_data_science")
    results  = filtered.invoke("what tools are available?")
    print(f"  Retrieved {len(results)} results scoped to Python docs only")
  except Exception as e:
    # Filter syntax varies by ChromaDB version — fall back to post-filter
    print(f"  [Note] ChromaDB filter syntax error ({e}) — using post-filter fallback")
    results = [r for r in vectorstore.similarity_search("what tools are available?", k=6)
                if "python" in r.metadata.get("source", "")]
    print(f"  Post-filter: {len(results)} results from Python docs")

  # ---- 6. Query Routing ─────────────────────────────────────────────────
  print("\n" + "-" * 50)
  print("TECHNIQUE 7: Query Routing")
  print("-" * 50)
  for q in [
    "How do I use pandas for data analysis?",
    "What is gradient boosting?",
    "How does the attention mechanism work in transformers?",
  ]:
    query_router.invoke(q)

  # ---- 7. Evaluation Metrics ────────────────────────────────────────────
  print("\n" + "-" * 50)
  print("TECHNIQUE 8: Retrieval Evaluation Metrics")
  print("-" * 50)
  test_query   = "What are techniques to prevent overfitting?"
  ground_truth = "L1 regularization, L2 regularization, dropout, early stopping, cross-validation"
  retrieved    = vectorstore.similarity_search(test_query, k=3)
  metrics      = evaluate_retrieval(test_query, retrieved, ground_truth)
  print(f"  Query:   '{test_query}'")
  print(f"  Metrics: {metrics}")
  print("\n  Production: use RAGAS for LLM-as-judge faithfulness scoring")
  print("    from ragas.metrics import faithfulness, answer_relevancy")
  print("    result = evaluate(dataset, metrics=[faithfulness, answer_relevancy])")

  print("\n" + "=" * 60)
  print("  ADVANCED DEMO COMPLETE")
  print("=" * 60)

In [13]:
run_advanced_rag_demo()


  ADVANCED RAG TECHNIQUES DEMO

[SETUP] Loading and chunking documents...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[LLM] Loading Qwen3 0.6B Model

--------------------------------------------------
TECHNIQUE 1: Hybrid Search (BM25 + Dense)
--------------------------------------------------

[HYBRID] Building BM25 + Dense hybrid retriever...
[HYBRID] Hybrid retriever ready (BM25 + Dense)
  Retrieved 4 docs (BM25 catches exact 'RMSE' match)
  - [llm_and_rag.txt] Vector Databases
Vector databases store and index high-dimensional embeddings fo...
  - [llm_and_rag.txt] LLM Evaluation
Evaluating LLMs is challenging due to the open-ended nature of te...

--------------------------------------------------
TECHNIQUE 2: MMR — Diverse Retrieval
--------------------------------------------------

[MMR] Building MMR retriever for diverse results...
[MMR] MMR retriever ready (k=4, λ=0.5)
  Retrieved 4 diverse chunks:
  - [ml_fundamentals.txt] Machine Learning Fundamentals

Supervised Learning
Supervised learning...
  - [ml_fundamentals.txt] Key supervised learning algorithms include:
- Decision Trees: Tree-lik.

In [14]:
%watermark

UsageError: Line magic function `%watermark` not found.
